In [ ]:
# !pip install psycopg2-binary -q

In [ ]:
import pandas as pd 
import geopandas as gpd 
import numpy as np 
from sqlalchemy import create_engine, text
import json
from openhexa.sdk import workspace

### Transform Provinces

In [ ]:
# Provinces
source_table = "cod_iaso_provinces"
target_table = "cod_iaso_provinces_ss"
simplification_tolerance = 0.01

# --- 1. Database Connection --- 
engine = create_engine(workspace.database_url)

# --- 2. Load the table ---
gdf_provinces = gpd.read_postgis(f'''SELECT * FROM public."{source_table}"''', con=engine, geom_col="geometry")

# --- 3. Drop rows with missing geometry ---
gdf = gdf_provinces.dropna(subset=["geometry"]).copy()

# --- 4. Extract first polygon --- 
def first_polygon_coords_list(geom):
    if geom.geom_type == "MultiPolygon":
        return list(geom.geoms[0].exterior.coords)
    elif geom.geom_type == "Polygon":
        return list(geom.exterior.coords)
    else:
        return list(geom.coords)  # points or lines
        
# --- 5. Convert to JSON string for Superset & Keep only relevant columns for Superset ---
gdf["geometry_simplified"] = gdf["geometry"].simplify(tolerance=simplification_tolerance, preserve_topology=True)
gdf["first_polygon"] = gdf["geometry_simplified"].apply(lambda g: json.dumps(first_polygon_coords_list(g), ensure_ascii=False))
# gdf["first_polygon"] = gdf["geometry"].apply(lambda g: json.dumps(first_polygon_coords_list(g), ensure_ascii=False))

# gdf["geo_json"] = gdf["geometry"].apply(lambda g: json.dumps(first_polygon_coords(g), ensure_ascii=False))
df_for_superset_provinces = gdf[["name", "ref", "parent", "id", "parent_ref", "first_polygon"]].rename(
    columns={"parent": "parent_name", "parent_ref": "parent_id", "name": "province", "ref": "province_id"}
)

df_for_superset_provinces.head(3)

In [ ]:
# Before pushing this table, drop the dependent views and recreate them after (check: integrated_malaria_dashboard_superset/dev/views_superset.ipynb): 
# - view vw_pnlp_palu_tot_prov_geometry 
# - view vw_indicators_proportion_prov 

# Push to DB
df_for_superset_provinces.to_sql(
    name=target_table,
    con=engine,
    schema="public",
    if_exists="replace",    
    index=False,
)

### Transform ZS

In [ ]:
# Zones de sante
source_table = "cod_iaso_zone_de_sante"
target_table = "cod_iaso_zone_de_sante_ss"
simplification_tolerance = 0.01

# --- 1. Database Connection --- 
engine = create_engine(workspace.database_url)

# --- 2. Load the table ---
gdf_provinces = gpd.read_postgis(f'''SELECT * FROM public."{source_table}"''', con=engine, geom_col="geometry")

# --- 3. Drop rows with missing geometry ---
gdf = gdf_provinces.dropna(subset=["geometry"]).copy()

# --- 4. Extract first polygon --- 
def first_polygon_coords_list(geom):
    if geom.geom_type == "MultiPolygon":
        return list(geom.geoms[0].exterior.coords)
    elif geom.geom_type == "Polygon":
        return list(geom.exterior.coords)
    else:
        return list(geom.coords)  # points or lines
        
# --- 5. Convert to JSON string for Superset & Keep only relevant columns for Superset ---
gdf["geometry_simplified"] = gdf["geometry"].simplify(tolerance=simplification_tolerance, preserve_topology=True)
gdf["first_polygon"] = gdf["geometry_simplified"].apply(lambda g: json.dumps(first_polygon_coords_list(g), ensure_ascii=False))
# gdf["first_polygon"] = gdf["geometry"].apply(lambda g: json.dumps(first_polygon_coords_list(g), ensure_ascii=False))

# gdf["geo_json"] = gdf["geometry"].apply(lambda g: json.dumps(first_polygon_coords(g), ensure_ascii=False))
df_for_superset_zs = gdf[["name", "ref", "parent", "parent_ref", "first_polygon"]].rename(
    columns={"parent": "province", "parent_ref": "province_id", "name": "zone_de_sante", "ref": "zone_de_sante_id"}
)

df_for_superset_zs.head(3)

In [ ]:
# import matplotlib.pyplot as plt

# geo_size_org = []
# geo_size_simp = []

# for _, row in gdf.iterrows():
#     geo_size_org.append(len(str(row["geometry"])))
#     geo_size_simp.append(len(str(row["geometry_simplified"])))

# # Plot
# plt.figure(figsize=(10, 5))
# plt.plot(geo_size_org, color="red", label="Original geometry size")
# plt.plot(geo_size_simp, color="blue", label="Simplified geometry size")
# plt.xlabel("Row index")
# plt.ylabel("String length of geometry")
# plt.title("Comparison of Original vs Simplified Geometry Sizes")
# plt.legend()
# plt.show()

In [ ]:
# Before pushing this table, drop the dependent views and recreate them after (see integrated_malaria_dashboard_superset/dev/views_superset.ipynb): 
# - view vw_indicators_proportion_zs 
# - view vw_pnlp_palu_tot_zs_geometry 

# Push to DB
df_for_superset_zs.to_sql(
    name=target_table,
    con=engine,
    schema="public",
    if_exists="replace",    
    index=False,
)